In [1]:
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [2]:
def create_data_model():
    """Stores the data for the problem."""
    data = {}
    data["distance_matrix"] = [
        # 0    1    2    3    4    5    6    7    8    9   10
        [  0, 548, 776, 696, 582, 274, 502, 194, 308, 194, 536],  # 0 depot
        [548,   0, 684, 308, 194, 502, 730, 354, 696, 742, 1084], # 1
        [776, 684,   0, 992, 878, 502, 274, 810,  468, 742, 400], # 2
        [696, 308, 992,   0, 114, 650, 878, 502, 844, 890, 1232], # 3
        [582, 194, 878, 114,   0, 536, 764, 388, 730, 776, 1118], # 4
        [274, 502, 502, 650, 536,   0, 228, 308, 194, 240, 582],  # 5
        [502, 730, 274, 878, 764, 228,   0, 536, 194, 468, 354],  # 6
        [194, 354, 810, 502, 388, 308, 536,   0, 342, 388, 730],  # 7
        [308, 696, 468, 844, 730, 194, 194, 342,   0, 274, 388],  # 8
        [194, 742, 742, 890, 776, 240, 468, 388, 274,   0, 342],  # 9
        [536,1084, 400,1232,1118, 582, 354, 730, 388, 342,   0],  # 10
    ]
    data["num_vehicles"] = 1
    data["depot"] = 0
    return data

In [3]:
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["depot"]
    )

    # Create Routing Model.
    routing = pywrapcp.RoutingModel(manager)


In [4]:
def distance_callback(from_index, to_index):
    """Returns the distance between the two nodes."""
    # Convert from routing variable Index to distance matrix NodeIndex.
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data["distance_matrix"][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
  

In [5]:
 # Define cost of each arc.
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [6]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

In [7]:

def print_solution(manager, routing, solution):
    """Prints solution on console."""
    print(f"Objective: {solution.ObjectiveValue()} miles")
    index = routing.Start(0)
    plan_output = "Route for vehicle 0:\n"
    route_distance = 0
    while not routing.IsEnd(index):
        plan_output += f" {manager.IndexToNode(index)} ->"
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
    plan_output += f" {manager.IndexToNode(index)}\n"
    plan_output += f"Route distance: {route_distance} miles\n"
    print(plan_output)

In [8]:
solution = routing.SolveWithParameters(search_parameters)
if solution:
    print_solution(manager, routing, solution)

Objective: 3104 miles
Route for vehicle 0:
 0 -> 9 -> 10 -> 2 -> 6 -> 8 -> 5 -> 3 -> 4 -> 1 -> 7 -> 0
Route distance: 3104 miles

